<a href="https://colab.research.google.com/github/aimtyaem/AGO/blob/codespace-ideal-winner-j9j74wxj7q4hq7gq/Gridstats.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install Tesseract OCR
!sudo apt update
!sudo apt install tesseract-ocr
!sudo apt install libtesseract-dev

# Install Python packages
!pip install pytesseract opencv-python pillow numpy pandas

# Import libraries
import cv2
import pytesseract
import numpy as np
from PIL import Image
import re
import pandas as pd
from google.colab import files

# Set Tesseract path
pytesseract.pytesseract.tesseract_cmd = '/usr/bin/tesseract'

# Upload the image file
uploaded = files.upload()
image_filename = list(uploaded.keys())[0]
print(f"Uploaded file: {image_filename}")

def extract_tiers(image_path):
    """Process the tiers image and extract classification ranges with flags"""
    try:
        img = Image.open(image_path)
    except Exception as e:
        raise ValueError(f"Error opening image: {str(e)}")

    # Preprocess image for OCR
    img_cv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]

    # Extract text and clean results
    text = pytesseract.image_to_string(thresh)
    lines = [line.strip().replace(',', '') for line in text.split('\n') if line.strip()]

    # Define tier classification with associated flags
    tiers = []
    for line in lines:
        try:
            if '+' in line:
                value = int(line.replace('+', ''))
                tiers.append({
                    'min': value,
                    'max': float('inf'),
                    'flag': 'critical_high'
                })
            elif '-' in line:
                if line.startswith('-'):
                    max_val = int(line)
                    tiers.append({
                        'min': -float('inf'),
                        'max': max_val,
                        'flag': 'fault'
                    })
                else:
                    parts = list(map(int, line.split('-')))
                    if parts[0] >= 100000:
                        flag = 'high_usage'
                    elif parts[0] >= 50000:
                        flag = 'medium_usage'
                    else:
                        flag = 'low_usage'
                    tiers.append({
                        'min': parts[0],
                        'max': parts[1],
                        'flag': flag
                    })
        except Exception as e:
            print(f"Skipping invalid line: {line} - {str(e)}")

    # Sort tiers descending by minimum value
    tiers.sort(key=lambda x: x['min'], reverse=True)
    return tiers

def classify_consumption(tiers, value):
    """Classify a consumption value into the appropriate tier"""
    for tier in tiers:
        if tier['min'] <= value <= tier['max']:
            return tier['flag']
    return 'unknown'

# Example usage
if __name__ == "__main__":
    # Load and process the tiers image
    tier_data = extract_tiers(image_filename)

    # Sample consumption values to classify
    test_values = [1500000, 750000, 300000, 150000, 75000, 30000, 15000, 5000, -500]

    # Classify and store results
    results = []
    for consumption in test_values:
        flag = classify_consumption(tier_data, consumption)
        results.append({
            'Consumption (kWh)': consumption,
            'Flag': flag
        })

    # Convert results to a DataFrame
    df = pd.DataFrame(results)

    # Save DataFrame to CSV
    csv_filename = 'consumption_classification.csv'
    df.to_csv(csv_filename, index=False)
    print(f"Results saved to {csv_filename}")

    # Download the CSV file
    files.download(csv_filename)

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
3 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr

Saving Electricity_consumption_per_country_map.png to Electricity_consumption_per_country_map (1).png
Uploaded file: Electricity_consumption_per_country_map (1).png
Results saved to consumption_classification.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>